# ETS MARL — HAPPO Compliant

**HAPPO-compliant reward & action space: direct bid pricing, no artificial guards**

This notebook is a testing playground for the ETS MARL simulation.
It works both in **Google Colab** (clones the repo) and **locally in VS Code**
(auto-detects project root). It runs tests, trains agents, and visualizes results.

Key changes from P14:
- Direct bid price action in [60, 500] (no markup mechanism)
- No sell gating (sell_coverage_floor removed)
- No coverage_penalty in reward
- No holding cost in reward (banking_holding_cost: 0.0)
- Shaping decays fully to zero
- Phase 2 obs: +coverage_ratio, +carry_forward_norm (7 extra dims)
- Penalty normalization: /100 (was /500)
- KL anchor decay: 8000 episodes (was 5000)

In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── Configuration ──────────────────────────────────────────────────────────
REPO_URL   = "https://github.com/DJH961/Thesis-Energy-Auction.git"
BRANCH     = "HAPPO_compliant"
PROJECT_SUBDIR = "ets_marl ppo"  # subdirectory containing src/ and configs/

def _is_project_root(path: Path) -> bool:
    return (path / "src").is_dir() and (path / "configs").is_dir()

def _running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

IN_COLAB = _running_in_colab()

if IN_COLAB:
    # ── Colab: clone repo and cd into project ─────────────────────────────
    os.chdir("/content")
    repo_name = Path(REPO_URL.rstrip("/")).name.replace(".git", "")
    clone_dir = Path("/content") / repo_name

    if clone_dir.exists():
        print(f"Removing existing clone: {clone_dir}")
        subprocess.run(["rm", "-rf", str(clone_dir)], check=False)

    clone_cmd = ["git", "clone", "-b", BRANCH, REPO_URL, str(clone_dir)]
    print("Cloning:", " ".join(clone_cmd))
    subprocess.run(clone_cmd, check=True)

    target = clone_dir / PROJECT_SUBDIR
    if not _is_project_root(target):
        raise FileNotFoundError(
            f"PROJECT_SUBDIR not valid: {target}\n"
            f"Repo contents: {os.listdir(clone_dir)}"
        )
    os.chdir(target)
else:
    # ── Local (VS Code / CLI): walk up to find project root ───────────────
    nb_dir = Path.cwd()
    candidates = [nb_dir] + list(nb_dir.parents)
    found = None
    for cand in candidates:
        if _is_project_root(cand):
            found = cand
            break
        sub = cand / PROJECT_SUBDIR
        if sub.is_dir() and _is_project_root(sub):
            found = sub
            break
    if found is None:
        raise FileNotFoundError(
            f"Cannot find project root (src/ + configs/) starting from {nb_dir}.\n"
            f"Make sure you open the notebook from within the repository."
        )
    os.chdir(found)

sys.path.insert(0, os.getcwd())
print(f"Running in: {'Colab' if IN_COLAB else 'Local (VS Code / CLI)'}")
print(f"Working directory: {os.getcwd()}")
print(f"Files: {sorted(os.listdir('.'))}")


## 1. Setup — Clone Repository & Install Dependencies

In [ ]:
# Install dependencies
if os.path.exists("requirements.txt"):
    !pip install -q -r requirements.txt
else:
    !pip install -q gymnasium numpy torch pyyaml pandas matplotlib seaborn pytest numpy-groupies

# Verify GPU availability
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# ── HAPPO-compliant verification: confirm source code is up to date ──
import yaml

# Check key signatures exist in source files
checks = {
    'src/agents/actor_critic.py': 'clamp',
    'src/agents/ppo_agent.py': 'RewardNormalizer',
    'src/environment/ets_environment.py': 'penalty_norm',
    'src/environment/company.py': 'coverage_ratio',
}

for path, keyword in checks.items():
    with open(path) as f:
        code = f.read()
    assert keyword in code, f'Missing {keyword!r} in {path}'
    print(f'\u2705 {path}: {keyword!r} found')

# Verify HAPPO-compliant config values
with open('configs/default.yaml') as f:
    _cfg = yaml.safe_load(f)

assert _cfg['ets']['reserve_price'] == 60.0
assert _cfg['auction']['price_min'] == 60.0
assert _cfg['auction']['price_max'] == 500.0
assert 'price_markup_low' not in _cfg['auction'], 'price_markup_low should be removed'
assert 'price_markup_high' not in _cfg['auction'], 'price_markup_high should be removed'
assert 'reference_anchor' not in _cfg['auction'], 'reference_anchor should be removed'
assert 'coverage_weight' not in _cfg.get('reward', {}), 'coverage_weight should be removed'
assert 'sell_coverage_floor' not in _cfg.get('trading', {}), 'sell_coverage_floor should be removed'
assert 'unsafe_sell_penalty_weight' not in _cfg.get('trading', {})
assert 'holding_cost_exponent' not in _cfg.get('trading', {})
assert _cfg['trading']['banking_holding_cost'] == 0.0
assert _cfg['reward']['shaping_weight_floor'] == 0.0
assert _cfg['ppo']['kl_anchor_decay_episodes'] == 8000
assert _cfg['ppo']['happo'] == True
assert _cfg['ppo']['centralized_critic'] == True
assert _cfg.get('hpp', {}).get('enabled', False)

print(f'\u2705 Direct bid price: [{_cfg["auction"]["price_min"]}, {_cfg["auction"]["price_max"]}]')
print(f'\u2705 banking_holding_cost: {_cfg["trading"]["banking_holding_cost"]}')
print(f'\u2705 shaping_weight_floor: {_cfg["reward"]["shaping_weight_floor"]}')
print(f'\u2705 kl_anchor_decay_episodes: {_cfg["ppo"]["kl_anchor_decay_episodes"]}')
print(f'\u2705 HPP enabled, pool_size={_cfg["hpp"]["pool_size"]}')
print()
print('\u2705 All HAPPO-compliant source code and config verified.')


## 2. Run Tests — Verify Everything Works

In [ ]:
!python -m pytest tests/ -v

## 3. Explore the Environment

In [ ]:
import os, yaml
print("CWD:", os.getcwd())

with open("configs/default.yaml") as f:
    config = yaml.safe_load(f)

print("Config keys:", list(config.keys()))

In [ ]:
import sys
sys.path.insert(0, '.')

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.environment.ets_environment import ETSEnvironment

with open('configs/default.yaml') as f:
    config = yaml.safe_load(f)

env = ETSEnvironment(config, seed=42)
obs1, _ = env.reset()

tech_names = config['technologies']['names']

print('=== Environment Summary ===')
print(f'Agents: {env.n_agents}')
print(f'Years: {env.n_years}')
print(f'Technologies: {tech_names}')
print(f'Phase 1 obs dim: {obs1.shape[1]}')
print(f'Initial cap: {env.cap_schedule.get_cap(0):.2f} Mt')
print()

for i, c in enumerate(env.companies):
    mix_str = ' | '.join([f'{tech_names[t]}: {c.mix[t]*100:.0f}%' for t in range(len(tech_names))])
    print(f'A{i+1}: {mix_str}')
    print(f'     Emissions: {c.compute_emissions():.2f} Mt | '
          f'Green: {c.green_frac*100:.0f}% | '
          f'Op. Cost: {c.compute_operational_cost():.0f} M€/yr | '
          f'Avg EF: {c.weighted_emission_factor:.3f} tCO2/MWh')


## 4. Run a Random-Policy Episode (Baseline)

In [ ]:
env = ETSEnvironment(config, seed=42)
obs1, _ = env.reset(seed=42)
n_agents = env.n_agents
rng = np.random.default_rng(42)

aq = config['auction']
inv = config['investment']
trading_cfg = config.get('trading', {})
qty_mult_low = aq.get('qty_mult_low', 0.3)
qty_mult_high = aq.get('qty_mult_high', 1.3)
sec_mult_low = trading_cfg.get('sec_mult_low', 0.8)
sec_mult_high = trading_cfg.get('sec_mult_high', 1.3)

year_data = []

for year in range(config['simulation']['n_years']):
    # Random auction actions (use config bounds)
    auction_actions = rng.uniform(
        [aq['price_min'], qty_mult_low, 0.0, -1.0, -1.0, -1.0],
        [aq['price_max'], qty_mult_high, inv['max_invest_frac'], 1.0, 1.0, 1.0],
        size=(n_agents, 6)
    ).astype(np.float32)

    obs2, _ = env.step_auction(auction_actions)

    # Random secondary actions (config-driven bounds)
    secondary_actions = rng.uniform(
        [sec_mult_low, -aq['quantity_max']], [sec_mult_high, aq['quantity_max']], size=(n_agents, 2)
    ).astype(np.float32)

    obs1, rewards, terminated, _, info = env.step_secondary(secondary_actions)
    log = info['year_log']

    row = {'year': year, 'cap': log['cap'], 'price': log['clearing_price']}
    for i in range(n_agents):
        row[f'green_A{i+1}'] = log['green_fracs'][i]
        row[f'emissions_A{i+1}'] = log['emissions'][i]
    year_data.append(row)

    if terminated:
        break

df = pd.DataFrame(year_data)
df

In [ ]:
# Plot: Green fraction evolution (random policy)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Green fracs
ax = axes[0]
for i in range(n_agents):
    ax.plot(df['year'], df[f'green_A{i+1}'] * 100, label=f'A{i+1}', marker='o', markersize=4)
ax.set_xlabel('Year')
ax.set_ylabel('Green Fraction (%)')
ax.set_title('Green Fraction (Random Policy)')
ax.legend()
ax.grid(True, alpha=0.3)

# Emissions
ax = axes[1]
for i in range(n_agents):
    ax.plot(df['year'], df[f'emissions_A{i+1}'], label=f'A{i+1}', marker='o', markersize=4)
ax.set_xlabel('Year')
ax.set_ylabel('Emissions (Mt CO₂)')
ax.set_title('Emissions (Random Policy)')
ax.legend()
ax.grid(True, alpha=0.3)

# Cap & Price
ax = axes[2]
ax.plot(df['year'], df['cap'], label='Cap (Mt)', marker='s', color='red')
ax2 = ax.twinx()
ax2.plot(df['year'], df['price'], label='Price (€/t)', marker='^', color='blue')
ax.set_xlabel('Year')
ax.set_ylabel('Cap (Mt)', color='red')
ax2.set_ylabel('Price (€/t)', color='blue')
ax.set_title('Cap Trajectory & Carbon Price')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Train PPO Agents (Short Run)

In [ ]:
import copy
from scripts.train import train_one_seed

train_config = copy.deepcopy(config)

# Short run overrides
train_config['simulation']['n_episodes'] = 1500
train_config['logging']['log_interval'] = 49
train_config['logging']['save_interval'] = 100

# HAPPO-compliant config verification
assert train_config['ppo']['centralized_critic'] == True
assert train_config['ppo']['happo'] == True
assert train_config['agent_cycling']['enabled'] == False
assert train_config['auction']['price_min'] == 60.0
assert train_config['auction']['price_max'] == 500.0
assert 'price_markup_low' not in train_config['auction']
assert train_config['trading']['banking_holding_cost'] == 0.0
assert train_config['reward']['shaping_weight_floor'] == 0.0
assert train_config.get('hpp', {}).get('enabled', False)

print('Quick run config check')
print('happo              :', train_config['ppo']['happo'])
print('centralized_critic :', train_config['ppo']['centralized_critic'])
print('agent_cycling      :', train_config['agent_cycling']['enabled'])
print('bid price range    :', train_config['auction']['price_min'], '-', train_config['auction']['price_max'])
print('holding_cost       :', train_config['trading']['banking_holding_cost'])
print('shaping_floor      :', train_config['reward']['shaping_weight_floor'])
print('kl_decay_eps       :', train_config['ppo']['kl_anchor_decay_episodes'])
print('hpp_enabled        :', train_config['hpp']['enabled'])
print('hpp_swap_prob      :', train_config['hpp']['swap_prob'])
print('entropy_coef       :', train_config['ppo']['entropy_coef'])
print('entropy_coef_final :', train_config['ppo']['entropy_coef_final'])
print('epsilon_start      :', train_config['exploration']['epsilon_start'])
print('epsilon_final      :', train_config['exploration']['epsilon_final'])
print('shaping_beta       :', train_config['reward']['shaping_beta'])

train_one_seed(train_config, seed=42)


## 6. Analyze Training Results

In [ ]:
# Load training logs
ep_df = pd.read_csv('results/training_log_s42.csv')
yr_df = pd.read_csv('results/year_log_s42.csv')

print(f'Training episodes: {len(ep_df)}')
print(f'Year-level records: {len(yr_df)}')
ep_df.tail()

In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
n_agents = config['companies']['n_agents']
window = max(1, len(ep_df) // 50)  # smoothing window

# 1. Rewards
ax = axes[0, 0]
for i in range(n_agents):
    col = f'reward_A{i+1}'
    ax.plot(ep_df['episode'], ep_df[col].rolling(window).mean(), label=f'A{i+1}', alpha=0.8)
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('Rewards (smoothed)')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Green fractions
ax = axes[0, 1]
for i in range(n_agents):
    col = f'green_frac_A{i+1}'
    ax.plot(ep_df['episode'], ep_df[col].rolling(window).mean() * 100, label=f'A{i+1}', alpha=0.8)
ax.set_xlabel('Episode')
ax.set_ylabel('Final Green (%)')
ax.set_title('Green Fraction at End of Episode')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Clearing price
ax = axes[1, 0]
ax.plot(ep_df['episode'], ep_df['clearing_price_last'].rolling(window).mean(), color='darkblue')
ax.set_xlabel('Episode')
ax.set_ylabel('€/t')
ax.set_title('Final Year Clearing Price')
ax.grid(True, alpha=0.3)

# 4. Actor loss
ax = axes[1, 1]
for i in range(n_agents):
    col = f'actor_loss_A{i+1}'
    ax.plot(ep_df['episode'], ep_df[col].rolling(window).mean(), label=f'A{i+1}', alpha=0.7)
ax.set_xlabel('Episode')
ax.set_ylabel('Loss')
ax.set_title('Actor Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Year-level analysis: last 50 episodes
last_eps = yr_df['episode'].unique()[-50:]
recent = yr_df[yr_df['episode'].isin(last_eps)]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Emissions by year (averaged over recent episodes)
ax = axes[0]
for i in range(n_agents):
    avg = recent.groupby('year')[f'emissions_A{i+1}'].mean()
    ax.plot(avg.index, avg.values, label=f'A{i+1}', marker='o', markersize=4)
ax.set_xlabel('Year')
ax.set_ylabel('Emissions (Mt)')
ax.set_title('Avg Emissions (last 50 episodes)')
ax.legend()
ax.grid(True, alpha=0.3)

# Green frac by year
ax = axes[1]
for i in range(n_agents):
    avg = recent.groupby('year')[f'green_frac_A{i+1}'].mean()
    ax.plot(avg.index, avg.values * 100, label=f'A{i+1}', marker='o', markersize=4)
ax.set_xlabel('Year')
ax.set_ylabel('Green (%)')
ax.set_title('Avg Green Fraction (last 50 episodes)')
ax.legend()
ax.grid(True, alpha=0.3)

# Carbon price trajectory
ax = axes[2]
avg_price = recent.groupby('year')['clearing_price'].mean()
avg_cap = recent.groupby('year')['cap'].mean()
ax.plot(avg_price.index, avg_price.values, label='Clearing Price', marker='^', color='blue')
ax2 = ax.twinx()
ax2.plot(avg_cap.index, avg_cap.values, label='Cap', marker='s', color='red', alpha=0.7)
ax.set_xlabel('Year')
ax.set_ylabel('Price (€/t)', color='blue')
ax2.set_ylabel('Cap (Mt)', color='red')
ax.set_title('Avg Price & Cap (last 50 episodes)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6b. P5–P8 & MAC Stochastic Deviation Analysis

Visualise the **random noise** and **abatement mechanisms** across training:
- **P5** — Emission demand shocks (ε ~ correlated normal, σ=7%)
- **P6** — Green capacity-factor noise + project cancellations
- **P8** — Banking holding costs (penalty for excess allowance hoarding)
- **MAC** — Fuel-switching abatement (coal→gas dispatch when carbon price > €65/t)

In [ ]:
# ── P5/P6/P8 stochastic deviation analysis ─────────────────────────────────
yr_df = pd.read_csv('results/year_log_s42.csv')
n_agents = config['companies']['n_agents']
tech_names = config['technologies']['names']

# Check which deviation columns exist
shock_cols    = [f'emission_shock_A{i+1}' for i in range(n_agents)]
cf_cols       = [f'cf_shock_A{i+1}'       for i in range(n_agents)]
cancel_cols   = [f'cancellation_A{i+1}'   for i in range(n_agents)]
hcost_cols    = [f'holding_cost_A{i+1}'   for i in range(n_agents)]
mac_cols      = [f'mac_reduction_A{i+1}'  for i in range(n_agents)]

has_deviations = all(c in yr_df.columns for c in shock_cols)
has_mac = all(c in yr_df.columns for c in mac_cols)

cmap = plt.get_cmap('tab10')
agent_colors = [cmap(i % 10) for i in range(n_agents)]
leg_ncol = max(1, n_agents // 4)
bar_alpha = max(0.25, 0.55 - n_agents * 0.03)  # reduce opacity with more agents

if not has_deviations:
    print('Deviation columns not found — re-run training with the updated train.py')
else:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # ── 1. P5 emission shock distribution (all agents, all years) ────────────
    ax = axes[0, 0]
    for i in range(n_agents):
        vals = yr_df[shock_cols[i]] * 100
        ax.hist(vals, bins=40, alpha=bar_alpha, label=f'A{i+1}',
                density=True, color=agent_colors[i])
    ax.axvline(0, color='k', lw=1, ls='--')
    ax.set_xlabel('Emission shock ε (%)')
    ax.set_ylabel('Density')
    ax.set_title('P5 — Emission Demand Shocks\n(all years & episodes)')
    ax.legend(fontsize=7, ncol=leg_ncol)
    ax.grid(True, alpha=0.3)

    # ── 2. P5 shock correlation check (A1 vs A(n//2) — spread archetypes) ────
    ax = axes[0, 1]
    idx_a, idx_b = 0, max(1, n_agents // 2)
    ax.scatter(yr_df[shock_cols[idx_a]] * 100, yr_df[shock_cols[idx_b]] * 100,
               alpha=0.15, s=4, color='steelblue')
    lims = [-25, 25]
    ax.plot(lims, lims, 'k--', lw=0.8, label='1:1')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel(f'A{idx_a+1} shock (%)')
    ax.set_ylabel(f'A{idx_b+1} shock (%)')
    corr = yr_df[shock_cols[idx_a]].corr(yr_df[shock_cols[idx_b]])
    ax.set_title(f'P5 — Cross-Agent Shock Correlation\nA{idx_a+1} vs A{idx_b+1}  (ρ_obs = {corr:.2f})')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # ── 3. MAC fuel-switching over training ──────────────────────────────────
    ax = axes[0, 2]
    if has_mac:
        ep_mac = (
            yr_df.groupby('episode')[[*mac_cols]]
            .sum()
            .rolling(50, min_periods=1)
            .mean()
        )
        for i in range(n_agents):
            ax.plot(ep_mac.index, ep_mac[mac_cols[i]],
                    label=f'A{i+1}', alpha=0.8, color=agent_colors[i])
        ax.set_xlabel('Episode')
        ax.set_ylabel('MAC abatement / episode (Mt)')
        ax.set_title('MAC Fuel-Switching over Training\n(50-ep rolling mean)')
        ax.legend(fontsize=7, ncol=leg_ncol)
    else:
        # Fallback: P5 shock volatility
        ep_shock_std = (
            yr_df.groupby('episode')[[*shock_cols]]
            .std()
            .rolling(50, min_periods=1)
            .mean()
        )
        for i in range(n_agents):
            ax.plot(ep_shock_std.index, ep_shock_std[shock_cols[i]] * 100,
                    label=f'A{i+1}', alpha=0.8, color=agent_colors[i])
        ax.axhline(7.0, color='k', ls='--', lw=0.8, label='σ_demand=7%')
        ax.set_xlabel('Episode')
        ax.set_ylabel('Within-episode shock std (%)')
        ax.set_title('P5 — Emission Shock Volatility over Training')
        ax.legend(fontsize=7, ncol=leg_ncol)
    ax.grid(True, alpha=0.3)

    # ── 4. P6 CF noise distribution (green techs averaged) ───────────────────
    ax = axes[1, 0]
    for i in range(n_agents):
        vals = yr_df[cf_cols[i]] * 100
        ax.hist(vals, bins=40, alpha=bar_alpha, label=f'A{i+1}',
                density=True, color=agent_colors[i])
    ax.axvline(0, color='k', lw=1, ls='--')
    ax.set_xlabel('Mean green CF noise (%)')
    ax.set_ylabel('Density')
    ax.set_title('P6 — Capacity-Factor Noise\n(mean across onshore/offshore/solar)')
    ax.legend(fontsize=7, ncol=leg_ncol)
    ax.grid(True, alpha=0.3)

    # ── 5. P6 project cancellations over training ─────────────────────────────
    ax = axes[1, 1]
    ep_cancels = (
        yr_df.groupby('episode')[[*cancel_cols]]
        .sum()
        .rolling(50, min_periods=1)
        .mean()
    )
    ep_cancels['total'] = ep_cancels[cancel_cols].sum(axis=1)
    ax.fill_between(ep_cancels.index, ep_cancels['total'],
                    alpha=0.3, color='coral', label='Total (all agents)')
    for i in range(n_agents):
        ax.plot(ep_cancels.index, ep_cancels[cancel_cols[i]],
                lw=0.8, label=f'A{i+1}', alpha=0.8, color=agent_colors[i])
    ax.set_xlabel('Episode')
    ax.set_ylabel('Avg cancellations / episode')
    ax.set_title('P6 — Project Cancellations over Training\n(50-ep rolling mean)')
    ax.legend(fontsize=7, ncol=leg_ncol)
    ax.grid(True, alpha=0.3)

    # ── 6. P8 holding cost over training ─────────────────────────────────────
    ax = axes[1, 2]
    has_hcost = all(c in yr_df.columns for c in hcost_cols)
    if has_hcost:
        ep_hcost = (
            yr_df.groupby('episode')[[*hcost_cols]]
            .sum()
            .rolling(50, min_periods=1)
            .mean()
        )
        for i in range(n_agents):
            ax.plot(ep_hcost.index, ep_hcost[hcost_cols[i]],
                    label=f'A{i+1}', alpha=0.8, color=agent_colors[i])
        ax.set_xlabel('Episode')
        ax.set_ylabel('Holding cost / episode (M€)')
        ax.set_title('P8 — Excess-Bank Holding Costs over Training\n(50-ep rolling mean)')
        ax.legend(fontsize=7, ncol=leg_ncol)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'holding_cost columns\nnot found in CSV',
                ha='center', va='center', transform=ax.transAxes)

    plt.suptitle('P5–P8 & MAC Stochastic Deviation Diagnostics', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

    # Summary statistics table
    print('\n─── P5/P6/P8/MAC deviation summary (all training data) ───')
    summary = {}
    for i in range(n_agents):
        row = {
            'ε_mean (%)':       round(yr_df[shock_cols[i]].mean()  * 100, 3),
            'ε_std (%)':        round(yr_df[shock_cols[i]].std()   * 100, 3),
            'CF_noise_std (%)': round(yr_df[cf_cols[i]].std()      * 100, 3),
            'cancels_total':    int(yr_df[cancel_cols[i]].sum()),
            'holdcost_sum (M€)': round(yr_df[hcost_cols[i]].sum(), 2) if has_hcost else 'n/a',
        }
        if has_mac:
            row['mac_total (Mt)'] = round(yr_df[mac_cols[i]].sum(), 2)
        summary[f'A{i+1}'] = row
    print(pd.DataFrame(summary).T.to_string())


## 6c. Year-by-Year Trajectory Analysis (last N episodes)

Slices the **year_log** to show how behaviour evolves *within* an episode across years 0–9.
Uses the last 100 episodes so the plots represent mature policy behaviour, not early exploration.

Four panels per row:
1. **Allocation vs Emissions** — over-allocation → surplus banking; under-allocation → shortfall risk  
2. **Banking** — bank_start and bank_end per year to see carry-forward dynamics  
3. **Secondary market net** — positive = net seller (revenue), negative = net buyer (cost)  
4. **Compliance surplus / shortfall** — pre-compliance headroom; negative → penalty risk

In [ ]:
# ── 6c: Year-by-year trajectory analysis ──────────────────────────────────
yr_df = pd.read_csv('results/year_log_s42.csv')
n_agents = config['companies']['n_agents']

N_LAST = 100  # episodes to average over
last_eps = yr_df['episode'].unique()[-N_LAST:]
recent = yr_df[yr_df['episode'].isin(last_eps)].copy()

# Check which new columns are present
has_bank_end   = 'bank_end_A1'            in recent.columns
has_sec_net    = 'secondary_net_A1'       in recent.columns
has_surplus    = 'compliance_surplus_A1'  in recent.columns
has_auc_cost   = 'auction_cost_A1'        in recent.columns

cmap = plt.get_cmap('tab10')
agent_colors = [cmap(i) for i in range(n_agents)]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ── Panel 1: Allocation vs Emissions ──────────────────────────────────────
ax = axes[0, 0]
for i in range(n_agents):
    alloc_avg = recent.groupby('year')[f'alloc_A{i+1}'].mean()
    emiss_avg = recent.groupby('year')[f'emissions_A{i+1}'].mean()
    ax.plot(alloc_avg.index, alloc_avg.values,
            color=agent_colors[i], lw=2, label=f'A{i+1} alloc')
    ax.plot(emiss_avg.index, emiss_avg.values,
            color=agent_colors[i], lw=1.5, ls='--', alpha=0.7)
ax.set_xlabel('Year in Episode')
ax.set_ylabel('Mt CO₂')
ax.set_title(f'Allocation (solid) vs Emissions (dashed)\n(avg last {N_LAST} eps)')
ax.legend(fontsize=6, ncol=2)
ax.grid(True, alpha=0.3)

# ── Panel 2: Banking dynamics (bank_start and bank_end) ────────────────────
ax = axes[0, 1]
for i in range(n_agents):
    bs = recent.groupby('year')[f'bank_start_A{i+1}'].mean()
    ax.plot(bs.index, bs.values, color=agent_colors[i], lw=2,
            label=f'A{i+1}')
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel('Year in Episode')
ax.set_ylabel('Banked Allowances (Mt)')
ax.set_title(f'Banking Dynamics (bank_start)\n(avg last {N_LAST} eps)')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)

# ── Panel 3: Secondary market net (positive = net revenue) ────────────────
ax = axes[1, 0]
if has_sec_net:
    bar_width = 0.8 / n_agents
    for i in range(n_agents):
        avg = recent.groupby('year')[f'secondary_net_A{i+1}'].mean()
        ax.bar(avg.index + i * bar_width - 0.4 + bar_width/2, avg.values,
               width=bar_width, color=agent_colors[i], alpha=0.75, label=f'A{i+1}')
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xlabel('Year in Episode')
    ax.set_ylabel('Net secondary P&L (M€)  +ve=revenue')
    ax.set_title(f'Secondary Market Net Revenue by Year\n(avg last {N_LAST} eps)')
    ax.legend(fontsize=6, ncol=2)
else:
    for i in range(n_agents):
        avg = recent.groupby('year')[f'trade_cost_A{i+1}'].mean()
        ax.plot(avg.index, -avg.values, color=agent_colors[i], lw=1.5,
                label=f'A{i+1}', marker='o', markersize=3)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xlabel('Year in Episode')
    ax.set_ylabel('−trade_cost (M€)  +ve=revenue')
    ax.set_title(f'Secondary Net (from trade_cost, last {N_LAST} eps)')
    ax.legend(fontsize=6, ncol=2)
ax.grid(True, alpha=0.3)

# ── Panel 4: Compliance surplus / shortfall ───────────────────────────────
ax = axes[1, 1]
if has_surplus:
    for i in range(n_agents):
        avg = recent.groupby('year')[f'compliance_surplus_A{i+1}'].mean()
        ax.plot(avg.index, avg.values, color=agent_colors[i], lw=2,
                marker='s', markersize=3, label=f'A{i+1}')
    ax.axhline(0, color='k', lw=1.2, ls='--', label='break-even')
else:
    for i in range(n_agents):
        sf = recent.groupby('year')[f'shortfall_A{i+1}'].mean()
        ax.plot(sf.index, -sf.values, color=agent_colors[i], lw=2,
                marker='s', markersize=3, label=f'A{i+1}')
    ax.axhline(0, color='k', lw=1.2, ls='--')
ax.set_xlabel('Year in Episode')
ax.set_ylabel('Surplus Mt  (+ve = safe, −ve = shortfall)')
ax.set_title(f'Compliance Surplus/Shortfall by Year\n(avg last {N_LAST} eps)')
ax.legend(fontsize=6, ncol=2)
ax.grid(True, alpha=0.3)

plt.suptitle(f'Year-by-Year Trajectory (last {N_LAST} episodes)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# ── Cost breakdown summary per year (if auction_cost logged) ──────────────
if has_auc_cost:
    print(f'\n── Average yearly cost breakdown (last {N_LAST} eps, all agents combined) ──')
    cols_show = {}
    for i in range(n_agents):
        cols_show[f'A{i+1}_auc(M€)'] = recent.groupby('year')[f'auction_cost_A{i+1}'].mean()
        cols_show[f'A{i+1}_inv(M€)'] = recent.groupby('year')[f'invest_cost_A{i+1}'].mean()
        cols_show[f'A{i+1}_pen(M€)'] = recent.groupby('year')[f'penalty_A{i+1}'].mean()
    print(pd.DataFrame(cols_show).round(2).to_string())


## 6d. Episode Consistency & Variance (ribbon plots)

Shows **mean ± 1 std** bands over the last 100 episodes for key year-level signals.
Wide bands → high variance / unstable policy. Narrow bands → consistent behaviour.

Includes a **price ribbon** (year 0→9 trajectory) and per-agent **green fraction trajectory**.

In [ ]:
# ── 6d: Episode consistency – ribbon plots (mean ± 1σ over last 100 episodes) ──
yr_df = pd.read_csv('results/year_log_s42.csv')
n_agents = config['companies']['n_agents']

N_LAST = 100
last_eps = yr_df['episode'].unique()[-N_LAST:]
recent = yr_df[yr_df['episode'].isin(last_eps)].copy()
years  = sorted(recent['year'].unique())

def ribbon(ax, grp, col, color, label):
    """Plot mean ± 1 std ribbon for `col` grouped by year."""
    mu  = grp[col].mean()
    std = grp[col].std().fillna(0)
    ax.plot(mu.index, mu.values, color=color, lw=2, label=label)
    ax.fill_between(mu.index, mu - std, mu + std,
                    color=color, alpha=0.18)

grp = recent.groupby('year')
# Use a colormap that handles 8+ agents
cmap = plt.get_cmap('tab10')
agent_colors = [cmap(i) for i in range(n_agents)]

fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# ── 1. Clearing price trajectory ──────────────────────────────────────────
ax = axes[0, 0]
ribbon(ax, grp, 'clearing_price', 'steelblue', 'Clearing Price')
ax.set_xlabel('Year'); ax.set_ylabel('€/t')
ax.set_title('Clearing Price Trajectory\nmean ± 1σ (last 100 eps)')
ax.grid(True, alpha=0.3); ax.legend(fontsize=8)

# ── 2. Green fraction per agent ───────────────────────────────────────────
ax = axes[0, 1]
for i in range(n_agents):
    ribbon(ax, grp, f'green_frac_A{i+1}', agent_colors[i], f'A{i+1}')
ax.set_xlabel('Year'); ax.set_ylabel('Green fraction')
ax.set_title('Green Fraction Trajectory\nmean ± 1σ')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x*100:.0f}%'))
ax.grid(True, alpha=0.3); ax.legend(fontsize=7, ncol=2)

# ── 3. Emissions per agent ────────────────────────────────────────────────
ax = axes[0, 2]
for i in range(n_agents):
    ribbon(ax, grp, f'emissions_A{i+1}', agent_colors[i], f'A{i+1}')
ax.set_xlabel('Year'); ax.set_ylabel('Mt CO₂')
ax.set_title('Realized Emissions\nmean ± 1σ (P5 shock visible as wide bands)')
ax.grid(True, alpha=0.3); ax.legend(fontsize=7, ncol=2)

# ── 4. Banking (bank_start) per agent ─────────────────────────────────────
ax = axes[1, 0]
for i in range(n_agents):
    ribbon(ax, grp, f'bank_start_A{i+1}', agent_colors[i], f'A{i+1}')
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel('Year'); ax.set_ylabel('Banked Mt')
ax.set_title('Banking Strategy\nmean ± 1σ')
ax.grid(True, alpha=0.3); ax.legend(fontsize=7, ncol=2)

# ── 5. Reward per agent ───────────────────────────────────────────────────
ax = axes[1, 1]
for i in range(n_agents):
    ribbon(ax, grp, f'reward_A{i+1}', agent_colors[i], f'A{i+1}')
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel('Year'); ax.set_ylabel('Reward')
ax.set_title('Per-Year Reward\nmean ± 1σ')
ax.grid(True, alpha=0.3); ax.legend(fontsize=7, ncol=2)

# ── 6. P8 holding cost (if logged) / shortfall ────────────────────────────
ax = axes[1, 2]
has_hcost = 'holding_cost_A1' in recent.columns
if has_hcost:
    for i in range(n_agents):
        ribbon(ax, grp, f'holding_cost_A{i+1}', agent_colors[i], f'A{i+1} hold_cost')
    ax.set_ylabel('Holding cost M€')
    ax.set_title('P8 Holding Costs by Year\nmean ± 1σ')
else:
    for i in range(n_agents):
        ribbon(ax, grp, f'shortfall_A{i+1}', agent_colors[i], f'A{i+1}')
    ax.set_ylabel('Shortfall Mt')
    ax.set_title('Shortfall by Year\nmean ± 1σ')
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel('Year'); ax.grid(True, alpha=0.3); ax.legend(fontsize=7, ncol=2)

plt.suptitle(f'Episode Consistency & Variance (last {N_LAST} episodes)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# ── Variance summary table ────────────────────────────────────────────────
print(f'\n── Year-level variance summary (std across {N_LAST} episodes, mean over years) ──')
var_summary = {}
for i in range(n_agents):
    var_summary[f'A{i+1}'] = {
        'price_std (€/t)':   round(recent.groupby('year')['clearing_price'].std().mean(), 2),
        'green_frac_std':    round(recent.groupby('year')[f'green_frac_A{i+1}'].std().mean(), 4),
        'emissions_std(Mt)': round(recent.groupby('year')[f'emissions_A{i+1}'].std().mean(), 3),
        'bank_std(Mt)':      round(recent.groupby('year')[f'bank_start_A{i+1}'].std().mean(), 3),
        'reward_std':        round(recent.groupby('year')[f'reward_A{i+1}'].std().mean(), 4),
    }
print(pd.DataFrame(var_summary).T.to_string())


## 7. Evaluate Best Agents (Deterministic)

In [ ]:
from scripts.train import build_agents

# Load best checkpoints
eval_env = ETSEnvironment(config, seed=0)
eval_agents = build_agents(eval_env, config, seed=0)
n_agents = config['companies']['n_agents']

ckpt_dir = 'results/checkpoints_s42'
for i, agent in enumerate(eval_agents):
    ckpt_path = os.path.join(ckpt_dir, f'agent_{i}_best.pt')
    if os.path.exists(ckpt_path):
        agent.load(ckpt_path)
        print(f'Loaded agent {i}')
    else:
        print(f'No checkpoint for agent {i} — using untrained policy')

# Run deterministic evaluation
obs1, _ = eval_env.reset(seed=0)
eval_data = []

for year in range(config['simulation']['n_years']):
    auction_actions = np.zeros((n_agents, 6), dtype=np.float32)
    for i in range(n_agents):
        action, _, _ = eval_agents[i].select_auction_action(obs1[i], deterministic=True)
        auction_actions[i] = action
    obs2, _ = eval_env.step_auction(auction_actions)

    secondary_actions = np.zeros((n_agents, 2), dtype=np.float32)
    for i in range(n_agents):
        action, _, _ = eval_agents[i].select_secondary_action(obs2[i], deterministic=True)
        secondary_actions[i] = action

    obs1, rewards, terminated, _, info = eval_env.step_secondary(secondary_actions)
    log = info['year_log']

    row = {'year': year, 'cap': log['cap'], 'price': log['clearing_price']}
    for i in range(n_agents):
        row[f'green_A{i+1}'] = log['green_fracs'][i]
        row[f'emissions_A{i+1}'] = log['emissions'][i]
        for t in range(5):
            row[f'mix_A{i+1}_{tech_names[t]}'] = log['tech_mixes'][i][t]
    eval_data.append(row)
    if terminated:
        break

eval_df = pd.DataFrame(eval_data)
green_cols = ['year', 'cap', 'price'] + [f'green_A{i+1}' for i in range(n_agents)]
print(eval_df[green_cols].to_string(index=False))

In [ ]:
# Technology mix evolution per agent
tech_names = config['technologies']['names']
colors = {'coal': '#555555', 'gas': '#FF8C00', 'onshore_wind': '#2E8B57',
          'offshore_wind': '#1E90FF', 'solar': '#FFD700'}

# Dynamic archetype labels (2 of each for 8 agents)
_base_archetypes = ['Coal-Heavy', 'Gas-Dominant', 'Mixed', 'Near-Green']
archetypes = []
for arch in _base_archetypes:
    count = 0
    for mix in config['companies']['initial_mix']:
        # Match archetype by green fraction
        green = sum(mix[j] for j, g in enumerate(config['technologies']['is_green']) if g)
        if arch == 'Coal-Heavy' and green < 0.25:
            count += 1
        elif arch == 'Gas-Dominant' and 0.25 <= green < 0.55:
            count += 1
        elif arch == 'Mixed' and 0.55 <= green < 0.85:
            count += 1
        elif arch == 'Near-Green' and green >= 0.85:
            count += 1
# Simpler: just assign based on position
archetypes = []
n_per = n_agents // 4
for arch in _base_archetypes:
    for k in range(max(1, n_per)):
        archetypes.append(arch)
# Pad if needed
while len(archetypes) < n_agents:
    archetypes.append(f'Agent {len(archetypes)+1}')

ncols = min(4, n_agents)
nrows = (n_agents + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows), sharey=True)
axes = np.atleast_1d(axes).flatten()

for i in range(n_agents):
    ax = axes[i]
    bottom = np.zeros(len(eval_df))
    for t in range(5):
        vals = eval_df[f'mix_A{i+1}_{tech_names[t]}'].values * 100
        ax.bar(eval_df['year'], vals, bottom=bottom,
               label=tech_names[t].replace('_', ' ').title(),
               color=colors[tech_names[t]], alpha=0.85)
        bottom += vals
    ax.set_xlabel('Year')
    if i % ncols == 0:
        ax.set_ylabel('Mix (%)')
    ax.set_title(f'A{i+1}: {archetypes[i]}')
    ax.set_ylim(0, 100)

# Hide unused axes
for j in range(n_agents, len(axes)):
    axes[j].set_visible(False)

# Single legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Technology Mix Evolution (Trained Policy)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Investment Cost Sanity Check

Verify that investment costs match real-world data (action plan §2).

In [ ]:
env = ETSEnvironment(config, seed=42)
env.reset()
c = env.companies[0]  # A1: coal-heavy

print('=== Investment Cost Sanity Check ===')
print(f'Company output: {c.output_twh} TWh/year = {c.output_mwh/1e6:.0f} million MWh')
print()

for tech_idx, name in [(2, 'Onshore Wind'), (3, 'Offshore Wind'), (4, 'Solar PV')]:
    frac = 0.03  # 3% of output
    delta_mwh = frac * c.output_mwh
    cf = c.capacity_factors[tech_idx]
    delta_mw = delta_mwh / (cf * 8760)
    cost = c.compute_investment_cost(tech_idx, frac)
    print(f'{name}:')
    print(f'  Shift: {frac*100:.0f}% = {delta_mwh/1e6:.0f} GWh/yr')
    print(f'  New capacity: {delta_mw:.0f} MW (at CF={cf:.0%})')
    print(f'  CapEx: {c.capex[tech_idx]:.0f} €/kW')
    print(f'  Investment cost: {cost:.1f} M€')
    print()

## 9. Full Training Run (Longer)

Uncomment and run for a more complete training. Takes ~15-30 min on Colab GPU.

In [ ]:
import copy
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from scripts.train import train_one_seed

full_config = copy.deepcopy(config)

# Full run overrides
full_config['simulation']['n_episodes'] = 30000
full_config['logging']['log_interval'] = 97
full_config['logging']['results_dir'] = 'results/'

# HAPPO-compliant verification
assert full_config['ppo']['happo'] == True
assert full_config['ppo']['centralized_critic'] == True
assert full_config['auction']['price_min'] == 60.0
assert full_config['auction']['price_max'] == 500.0
assert 'price_markup_low' not in full_config['auction']
assert full_config['trading']['banking_holding_cost'] == 0.0
assert full_config['reward']['shaping_weight_floor'] == 0.0
assert full_config.get('hpp', {}).get('enabled', False)

print('Full run config check')
print('happo              :', full_config['ppo']['happo'])
print('centralized_critic :', full_config['ppo']['centralized_critic'])
print('bid price range    :', full_config['auction']['price_min'], '-', full_config['auction']['price_max'])
print('holding_cost       :', full_config['trading']['banking_holding_cost'])
print('shaping_floor      :', full_config['reward']['shaping_weight_floor'])
print('kl_decay_eps       :', full_config['ppo']['kl_anchor_decay_episodes'])
print('hpp_enabled        :', full_config['hpp']['enabled'])
print('entropy_coef       :', full_config['ppo']['entropy_coef'])
print('entropy_coef_final :', full_config['ppo']['entropy_coef_final'])
print('epsilon_start      :', full_config['exploration']['epsilon_start'])
print('shaping_beta       :', full_config['reward']['shaping_beta'])

# ── Live loss plotting callback ──────────────────────────────────────────────
n_agents = full_config['companies']['n_agents']
_loss_fig, _loss_axes = plt.subplots(1, 2, figsize=(14, 4))
_loss_display = display(_loss_fig, display_id=True)
plt.close(_loss_fig)  # prevent duplicate static render

def _plot_losses(episode, csv_path):
    """Called every log_interval."""
    try:
        df = pd.read_csv(csv_path)
    except Exception:
        return

    for ax in _loss_axes:
        ax.clear()

    episodes = df['episode']

    ax_a = _loss_axes[0]
    for i in range(n_agents):
        col = f'actor_loss_A{i+1}'
        if col in df.columns:
            ax_a.plot(episodes, df[col], label=f'A{i+1}', linewidth=0.8)
    ax_a.set_title('Actor Loss')
    ax_a.set_xlabel('Episode')
    ax_a.set_ylabel('Loss')
    ax_a.legend(fontsize=7, ncol=4, loc='upper right')
    ax_a.grid(True, alpha=0.3)

    ax_c = _loss_axes[1]
    for i in range(n_agents):
        col = f'critic_loss_A{i+1}'
        if col in df.columns:
            ax_c.plot(episodes, df[col], label=f'A{i+1}', linewidth=0.8)
    ax_c.set_title('Critic Loss')
    ax_c.set_xlabel('Episode')
    ax_c.set_ylabel('Loss')
    ax_c.legend(fontsize=7, ncol=4, loc='upper right')
    ax_c.grid(True, alpha=0.3)

    _loss_fig.tight_layout()
    _loss_display.update(_loss_fig)

# Launch training with live plotting
train_one_seed(full_config, seed=42, on_log=_plot_losses)


In [ ]:
# Download results (Colab only — locally files are already on disk)
try:
    from google.colab import files
    files.download('results/training_log_s42.csv')
    files.download('results/year_log_s42.csv')
except ImportError:
    print('Running locally — results saved to:', os.path.abspath('results/'))
